# 2. LOBO Calibration on Held-out HC
- Same checks as `2_cv_analysis.ipynb`, applied to HC of a batch the engine never saw.
- Each held-out HC is compared with its own CV score (same sample, batch seen in training).
  - The gap between the two columns is the cost of an unseen batch.
- Detection performance on injected signals: `6_in_silico_method_comparison.ipynb`.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import json
import os
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde, kurtosis, norm, skew

os.environ["NM_PIPELINE"] = "p0"
sys.path.insert(0, str(Path.cwd().parents[1]))
import MixedEffectsModeling.config as config
from MixedEffectsModeling.core.calibration import bh_fdr_reject
from MixedEffectsModeling.notebookscripts.LOBO.lobo_engine import selected_batches
from viz_style import apply_style, FS_ANNOT_LG, FS_ANNOT_SM
apply_style()

LOBO = config.LOBO_DIR
CV = config.CV_DIR
FIG = LOBO / 'Figures'
FIG.mkdir(parents=True, exist_ok=True)

TARGET = {'mean': 0.0, 'std': 1.0, 'skew': 0.0, 'kurt': 0.0, 'exceed': 0.05}
Q_GENE = (0.01, 0.005)
Q_SAMPLE = 0.05
COL = {'CV (seen batch)': '#2b5c8f', 'LOBO (unseen batch)': '#d62828'}

## 0. Held-out HC and gene set
- HC of each held-out batch that pass the OOD filter (`ood_mask.npy`, written by `lobo_engine.py`).
- CV Z for the same samples come from `cv_zscores*.pkl`, whose arrays are the fold test sets concatenated in fold order (`cv_folds.json`).
- Genes that failed in any fold are dropped.

In [2]:
folds = json.load(open(config.OUTRIDER_COMPARISON_DIR / 'cv_folds.json'))
cv_order = np.concatenate([folds[k]['test'] for k in sorted(folds, key=int)])
cv_pos = pd.Series(np.arange(len(cv_order)), index=cv_order)
cv_raw = {g: z for g, z in pickle.load(open(CV / 'cv_zscores.pkl', 'rb')).items() if len(z) == len(cv_order)}
cv_sh = pickle.load(open(CV / 'cv_zscores_shash.pkl', 'rb'))

metas = selected_batches()
lobo_genes = [pickle.load(open(LOBO / m['batch_dir'] / 'gene_names.pkl', 'rb')) for m in metas]
GENES = sorted(set.intersection(*map(set, lobo_genes)) & set(cv_raw))

parts, rows = [], []
for m, g in zip(metas, lobo_genes):
    bdir = LOBO / m['batch_dir']
    keep = np.array(m['test_is_hc']) & np.load(bdir / 'ood_mask.npy')
    names = np.array(m['test_names'])[keep]
    gi = pd.Index(g).get_indexer(GENES)
    parts.append(dict(batch=m['batch_id'], names=names,
                      lobo_raw=np.load(bdir / 'Z_test.npy')[keep][:, gi].astype(float),
                      lobo_sh=np.load(bdir / 'Z_test_shash.npy')[keep][:, gi].astype(float)))
    rows.append(dict(batch=m['batch_id'], n_hc_test=int(np.sum(m['test_is_hc'])), n_hc_ood_kept=int(keep.sum())))

pos = cv_pos.loc[np.concatenate([p['names'] for p in parts])].values
BATCH = np.concatenate([np.repeat(p['batch'], len(p['names'])) for p in parts])
Z = {('CV (seen batch)', 'raw'): np.column_stack([cv_raw[g][pos] for g in GENES]).astype(float),
     ('CV (seen batch)', 'shash'): np.column_stack([cv_sh[g][pos] for g in GENES]).astype(float),
     ('LOBO (unseen batch)', 'raw'): np.vstack([p['lobo_raw'] for p in parts]),
     ('LOBO (unseen batch)', 'shash'): np.vstack([p['lobo_sh'] for p in parts])}
for k in Z:
    Z[k][~np.isfinite(Z[k])] = np.nan

print(f'genes shared by CV and all {len(metas)} LOBO fits: {len(GENES)}')
pd.DataFrame(rows)

genes shared by CV and all 5 LOBO fits: 19591


,batch,n_hc_test,n_hc_ood_kept
0,Chen et al._Batch_1,31,25
1,Moore et al._Batch_1,48,48
2,Moufarrej et al._Batch_1,54,48
3,Moufarrej et al._Batch_4,47,42
4,Ward Z et al._Batch_1,29,26


## A1. Per-gene Z moments (before / after SHASH)
- Mean, SD, skewness and excess kurtosis of each gene over the pooled held-out HC.
- Dashed line = ideal N(0, 1).

In [3]:
def moments(z):
    return pd.DataFrame({'mean': np.nanmean(z, 0), 'std': np.nanstd(z, 0),
                         'skew': skew(z, 0, nan_policy='omit'), 'kurt': kurtosis(z, 0, nan_policy='omit')},
                        index=GENES)


MOM = {k: moments(v) for k, v in Z.items()}

fig, axes = plt.subplots(2, 4, figsize=(11, 5))
for col, (stat, lab) in enumerate([('mean', 'Mean'), ('std', 'SD'), ('skew', 'Skewness'), ('kurt', 'Excess kurtosis')]):
    lo, hi = np.nanpercentile(np.concatenate([MOM[k][stat].values for k in MOM]), [0.5, 99.5])
    grid = np.linspace(lo, hi, 512)
    for row, kind in enumerate(['raw', 'shash']):
        ax = axes[row, col]
        for arm, c in COL.items():
            v = MOM[(arm, kind)][stat].dropna().values
            d = gaussian_kde(v)(grid)
            ax.fill_between(grid, d, color=c, alpha=0.25, lw=0)
            ax.plot(grid, d, color=c, lw=1.2, label=arm.split(' ')[0])
        ax.axvline(TARGET[stat], color='black', ls='--', lw=1.2)
        ax.set_xlim(lo, hi); ax.set_ylim(bottom=0)
        ax.set_title(f"{lab}, {'before' if kind == 'raw' else 'after'} SHASH", fontsize=FS_ANNOT_LG)
        ax.set_ylabel('Density' if col == 0 else '')
        ax.tick_params(labelsize=FS_ANNOT_SM)
        if (row, col) == (0, 0):
            ax.legend(frameon=False, fontsize=FS_ANNOT_SM, loc='upper left')
        ax.grid(True, ls=':', alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / 'lobo_moments_cv_vs_lobo.png', dpi=300, bbox_inches='tight')
plt.show()

pd.DataFrame({f'{arm} / {kind}': MOM[(arm, kind)].median() for arm, kind in MOM}).T.round(4)

/tmp/ipykernel_1005182/2175792271.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,mean,std,skew,kurt
CV (seen batch) / raw,0.0196,1.0112,-0.1242,0.2890
CV (seen batch) / shash,0.0246,1.0139,0.0155,0.2044
LOBO (unseen batch) / raw,-0.0049,1.0012,-0.1088,0.3625
LOBO (unseen batch) / shash,0.0146,1.0032,0.0853,0.2082


## A2. Exceedance calibration
- Left: fraction of HC x gene cells with two-sided p < alpha against alpha (diagonal = calibrated); SHASH Z, one line per held-out batch.
- Right: per-gene |Z| > 1.96 rate per batch (target 0.05).

In [4]:
ALPHAS = np.logspace(-4, np.log10(0.5), 25)
batches = list(dict.fromkeys(BATCH))


def exceed_curve(z):
    p = 2 * norm.sf(np.abs(z[np.isfinite(z)]))
    return np.array([(p < a).mean() for a in ALPHAS])


fig, (ax, bx) = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={'width_ratios': [1, 1.4]})
for arm, c in COL.items():
    for b in batches:
        ax.plot(ALPHAS, exceed_curve(Z[(arm, 'shash')][BATCH == b]), color=c, lw=1, alpha=0.7)
    ax.plot([], [], color=c, label=arm)
ax.plot(ALPHAS, ALPHAS, color='black', ls='--', lw=1)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('nominal alpha'); ax.set_ylabel('observed exceedance')
ax.set_title('Exceedance vs nominal (SHASH)', fontsize=FS_ANNOT_LG)
ax.legend(frameon=False, fontsize=FS_ANNOT_SM)

rate = {arm: [100 * np.nanmean(np.abs(Z[(arm, 'shash')][BATCH == b]) > 1.96, 0) for b in batches] for arm in COL}
x = np.arange(len(batches)); w = 0.38
for k, (arm, c) in enumerate(COL.items()):
    bp = bx.boxplot(rate[arm], positions=x + (k - 0.5) * w, widths=w * 0.9, showfliers=False, patch_artist=True,
                    medianprops=dict(color='black', lw=2.2))
    for b_ in bp['boxes']:
        b_.set_facecolor(c); b_.set_alpha(0.5)
    bx.plot([], [], color=c, lw=6, alpha=0.5, label=arm)
bx.axhline(100 * TARGET['exceed'], color='black', ls='--', lw=1)
bx.set_ylim(0, 50)
bx.set_xticks(x); bx.set_xticklabels([b.replace(' et al._Batch_', '_') for b in batches], rotation=45, ha='right')
bx.set_ylabel('% of samples with |Z| > 1.96 per gene'); bx.set_title('Per-gene exceedance by batch', fontsize=FS_ANNOT_LG)
bx.legend(frameon=False, fontsize=FS_ANNOT_SM)
plt.tight_layout()
plt.savefig(FIG / 'lobo_exceedance_calibration.png', dpi=300, bbox_inches='tight')
plt.show()

/tmp/ipykernel_1005182/2814896445.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## A3. BH outliers
- Left: per gene, BH across the pooled held-out HC (as in `2_cv_analysis.ipynb`); number of outlier samples, SHASH Z.
- Right two panels: per sample, BH across genes (q = 0.05).
  - Number of significant genes as a count and as a percentage of all tested genes.

In [5]:
CAT_LABELS = ['0 (Best)', '1', '2', '3', '4', '5+']


def gene_outliers(z, q):
    out = np.zeros(z.shape[1], int)
    for j in range(z.shape[1]):
        v = z[:, j][np.isfinite(z[:, j])]
        out[j] = bh_fdr_reject(2 * norm.sf(np.abs(v)), q=q).sum() if len(v) else 0
    return out


def sample_nsig(z, q):
    return np.array([bh_fdr_reject(2 * norm.sf(np.abs(r[np.isfinite(r)])), q=q).sum() for r in z])


NOUT = {(arm, q): gene_outliers(Z[(arm, 'shash')], q) for arm in COL for q in Q_GENE}
NSIG = {arm: sample_nsig(Z[(arm, 'shash')], Q_SAMPLE) for arm in COL}

fig, axes = plt.subplots(1, len(Q_GENE) + 2, figsize=(17, 3.8), gridspec_kw={'width_ratios': [1, 1, 1.3, 1.3]})
x = np.arange(len(CAT_LABELS)); w = 0.38
for ax, q in zip(axes, Q_GENE):
    for k, (arm, c) in enumerate(COL.items()):
        n = pd.cut(np.clip(NOUT[(arm, q)], 0, 5), bins=[-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5], labels=CAT_LABELS)
        vals = pd.Series(n).value_counts().reindex(CAT_LABELS).fillna(0).values
        bars = ax.bar(x + (k - 0.5) * w, vals, w, color=c, edgecolor='black', label=arm)
        for b_ in bars:
            if b_.get_height() > 0:
                ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f'{int(b_.get_height())}',
                        ha='center', va='bottom', fontsize=FS_ANNOT_SM)
    ax.set_yscale('log'); ax.set_xticks(x); ax.set_xticklabels(CAT_LABELS)
    ax.set_xlabel('outlier samples per gene'); ax.set_ylabel('# genes (log)')
    ax.set_title(f'BH q={q:.3f}: outliers per gene', fontsize=FS_ANNOT_LG)
    ax.legend(frameon=False, fontsize=FS_ANNOT_SM)

def nsig_box(ax, scale):
    x = np.arange(len(batches))
    for k, (arm, c) in enumerate(COL.items()):
        data = [NSIG[arm][BATCH == b] * scale for b in batches]
        bp = ax.boxplot(data, positions=x + (k - 0.5) * w, widths=w * 0.9, showfliers=False, patch_artist=True,
                    medianprops=dict(color='black', lw=2.2))
        for b_ in bp['boxes']:
            b_.set_facecolor(c); b_.set_alpha(0.5)
        for i, d in enumerate(data):
            ax.scatter(np.full(len(d), x[i] + (k - 0.5) * w), d, s=6, color=c, alpha=0.6, zorder=3)
        ax.plot([], [], color=c, lw=6, alpha=0.5, label=arm)
    ax.set_xticks(x); ax.set_xticklabels([b.replace(' et al._Batch_', '_') for b in batches], rotation=45, ha='right')
    ax.legend(frameon=False, fontsize=FS_ANNOT_SM)


ax = axes[-2]
nsig_box(ax, 1)
ax.set_yscale('symlog', linthresh=1)
ax.set_ylim(bottom=0)
ax.set_ylabel(f'BH-significant genes (q={Q_SAMPLE})')
ax.set_title('Per-sample false calls (count)', fontsize=FS_ANNOT_LG)

ax = axes[-1]
nsig_box(ax, 100 / len(GENES))
ax.set_ylim(0, 100)
ax.set_ylabel(f'% of {len(GENES):,} tested genes')
ax.set_title('Per-sample false calls (%)', fontsize=FS_ANNOT_LG)
plt.tight_layout()
plt.savefig(FIG / 'lobo_bh_outliers.png', dpi=300, bbox_inches='tight')
plt.show()

/tmp/ipykernel_1005182/3368132626.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary per held-out batch

In [6]:
summ = []
for b in batches:
    r = BATCH == b
    for arm in COL:
        z = Z[(arm, 'shash')][r]
        summ.append(dict(batch=b, arm=arm, n_hc=int(r.sum()),
                         median_gene_mean=np.nanmedian(np.nanmean(z, 0)), median_gene_sd=np.nanmedian(np.nanstd(z, 0)),
                         exceed_196=np.nanmean(np.abs(z) > 1.96), exceed_329=np.nanmean(np.abs(z) > 3.29),
                         nsig_median=np.median(NSIG[arm][r]), nsig_mean=NSIG[arm][r].mean(),
                         nsig_pct_median=100 * np.median(NSIG[arm][r]) / len(GENES),
                         nsig_pct_mean=100 * NSIG[arm][r].mean() / len(GENES),
                         frac_samples_nsig0=(NSIG[arm][r] == 0).mean()))
summ = pd.DataFrame(summ)
summ.to_csv(LOBO / 'lobo_hc_calibration_summary.csv', index=False)
summ.pivot(index='batch', columns='arm').round(4)

n_hc                     median_gene_mean  \
arm                      CV (seen batch) LOBO (unseen batch)  CV (seen batch)   
batch                                                                           
Chen et al._Batch_1                   25                  25           0.0933   
Moore et al._Batch_1                  48                  48          -0.0940   
Moufarrej et al._Batch_1              48                  48           0.0406   
Moufarrej et al._Batch_4              42                  42           0.0637   
Ward Z et al._Batch_1                 26                  26           0.0192   

                                              median_gene_sd  \
arm                      LOBO (unseen batch) CV (seen batch)   
batch                                                          
Chen et al._Batch_1                  -0.0482          0.9496   
Moore et al._Batch_1                 -0.0556          1.0016   
Moufarrej et al._Batch_1              0.0306          0.9728   
Moufarrej et al._Batch_4              0.0729          0.9667   
Ward Z et al._Batch_1                 0.0373          0.9449   

                                                  exceed_196  \
arm                      LOBO (unseen batch) CV (seen batch)   
batch                                                          
Chen et al._Batch_1                   0.9145          0.0528   
Moore et al._Batch_1                  0.9823          0.0591   
Moufarrej et al._Batch_1              0.8737          0.0425   
Moufarrej et al._Batch_4              0.9544          0.0476   
Ward Z et al._Batch_1                 0.8494          0.0665   

                                                  exceed_329  \
arm                      LOBO (unseen batch) CV (seen batch)   
batch                                                          
Chen et al._Batch_1                   0.0462          0.0035   
Moore et al._Batch_1                  0.0619          0.0045   
Moufarrej et al._Batch_1              0.0446          0.0017   
Moufarrej et al._Batch_4              0.0540          0.0027   
Ward Z et al._Batch_1                 0.0437          0.0039   

                                                 nsig_median  \
arm                      LOBO (unseen batch) CV (seen batch)   
batch                                                          
Chen et al._Batch_1                   0.0036            12.0   
Moore et al._Batch_1                  0.0063            21.5   
Moufarrej et al._Batch_1              0.0027             3.0   
Moufarrej et al._Batch_4              0.0033            15.0   
Ward Z et al._Batch_1                 0.0032             8.0   

                                                   nsig_mean  \
arm                      LOBO (unseen batch) CV (seen batch)   
batch                                                          
Chen et al._Batch_1                      6.0         34.2000   
Moore et al._Batch_1                    35.5        262.5417   
Moufarrej et al._Batch_1                 5.0          7.2292   
Moufarrej et al._Batch_4                16.0         23.4524   
Ward Z et al._Batch_1                   11.0        318.1538   

                                             nsig_pct_median  \
arm                      LOBO (unseen batch) CV (seen batch)   
batch                                                          
Chen et al._Batch_1                  33.8800          0.0613   
Moore et al._Batch_1                 72.3750          0.1097   
Moufarrej et al._Batch_1             20.6458          0.0153   
Moufarrej et al._Batch_4            308.0952          0.0766   
Ward Z et al._Batch_1                37.1154          0.0408   

                                               nsig_pct_mean  \
arm                      LOBO (unseen batch) CV (seen batch)   
batch                                                          
Chen et al._Batch_1                   0.0306          0.1746   
Moore et al._Batch_1                  0.1812         

## B. Held-out disease vs same-batch HC (RKHS distance), engine and baselines
- Each held-out batch: OOD-kept HC and disease (Moore_1 disease = PDAC only), batches with >= 25 HC and >= 10 disease.
- Distance of each sample's Z vector to the mean kernel embedding of the batch's HC (RBF, median-heuristic bandwidth; each HC left out of its own reference).
- Engine: SHASH Z of every scored gene. Baselines: the frozen LOBO fits of `6_MethodComparison` (train = HC of the other batches) on their own gene sets (AE/PCA: fit genes; OutSingle/PEER: AE input genes).
- Upper half = HC, lower half = disease, line = median; stars = one-sided Mann-Whitney disease > HC, BH across batches within each method.

In [13]:
import scanpy as sc
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import mannwhitneyu

sys.path.insert(0, str(Path.cwd().parent / 'notebookscripts' / 'MethodComparison'))
import frozen_latent
import outsingle
import peer_arm
from MixedEffectsModeling.notebookscripts.LOBO.lobo_engine import load_full_data

DIS_SUBSET = {'Moore et al._Batch_1': ['PDAC']}
MIN_HC, MIN_DIS, Z_CAP = 25, 10, 10.0
ARM_LAB = {'engine': 'Normative model', 'autoencoder': 'OUTRIDER (AE)', 'pca': 'PCA', 'outsingle': 'OutSingle', 'peer': 'PEER'}
RKHS_PATH = LOBO / 'lobo_rkhs_distance_methods.csv'


def rkhs_distance(Z, is_hc):
    Z = np.clip(np.nan_to_num(Z, nan=0.0, posinf=Z_CAP, neginf=-Z_CAP), -Z_CAP, Z_CAP)
    d2 = pdist(Z, metric='sqeuclidean')
    gamma = 1.0 / np.median(d2[d2 > 0])
    H, D = Z[is_hc], Z[~is_hc]
    Khh = np.exp(-gamma * squareform(pdist(H, metric='sqeuclidean')))
    Kdh = np.exp(-gamma * cdist(D, H, metric='sqeuclidean'))
    n, row, tot = len(H), Khh.sum(1) - 1.0, Khh.sum()
    out = np.empty(len(Z))
    out[is_hc] = 1.0 - 2.0 / (n - 1) * row + (tot - 2 * row - 1.0) / (n - 1) ** 2
    out[~is_hc] = 1.0 - 2.0 / n * Kdh.sum(1) + tot / n ** 2
    return np.sqrt(np.clip(out, 0, None))


if RKHS_PATH.exists():
    rk = pd.read_csv(RKHS_PATH)
else:
    cond = sc.read_h5ad(config.H5AD_PATH, backed='r').obs['Stage/Condition'].astype(str)
    full = load_full_data()
    row = {n: i for i, n in enumerate(full['names'])}
    hc_counts = pd.read_csv(config.OUTRIDER_COMPARISON_DIR / 'hc_counts.csv', index_col=0)
    lfolds = json.load(open(config.METHOD_COMP_CACHE_DIR / 'lobo_folds.json'))
    parts = []
    for m in selected_batches():
        bdir = LOBO / m['batch_dir']
        names, is_hc = np.array(m['test_names']), np.array(m['test_is_hc'])
        keep = np.load(bdir / 'ood_mask.npy')
        if m['batch_id'] in DIS_SUBSET:
            keep &= is_hc | np.isin(cond.loc[names].values, DIS_SUBSET[m['batch_id']])
        names, is_hc = names[keep], is_hc[keep]
        counts = lambda genes: full['Y'][[row[n] for n in names]][:, [full['gene_col'][g] for g in genes]].astype(np.float64)
        Zs = {'engine': np.load(bdir / 'Z_test_shash.npy')[keep]}
        tag = f"lobo_{lfolds[m['batch_id']]['batch_dir']}"
        for impl in ['autoencoder', 'pca']:
            fit = frozen_latent.load_fit(impl, tag)
            Zs[impl] = frozen_latent.score_frozen(fit, counts(fit['genes']))['z']
        ae = frozen_latent.load_fit('autoencoder', tag)
        y_tr = hc_counts.loc[ae['genes'], lfolds[m['batch_id']]['train']].to_numpy(dtype=np.float64).T
        for arm, mod in [('outsingle', outsingle), ('peer', peer_arm)]:
            Zs[arm] = mod.score_frozen(mod.fit_train(y_tr, f'lobo_{tag}'), counts(ae['genes']))['z']
        parts += [pd.DataFrame(dict(arm=a, batch=m['batch_id'], sample=names, is_hc=is_hc, rkhs=rkhs_distance(z, is_hc)))
                  for a, z in Zs.items()]
    rk = pd.concat(parts, ignore_index=True)
    rk.to_csv(RKHS_PATH, index=False)

rk_stats = []
for (a, b), g in rk.groupby(['arm', 'batch']):
    h, d = g.loc[g.is_hc, 'rkhs'], g.loc[~g.is_hc, 'rkhs']
    if len(h) < MIN_HC or len(d) < MIN_DIS:
        continue
    u, p = mannwhitneyu(d, h, alternative='greater')
    rk_stats.append(dict(arm=a, batch=b, n_hc=len(h), n_dis=len(d), auc=u / (len(h) * len(d)), p=p))
rk_stats = pd.DataFrame(rk_stats)
def bh(p):
    p = np.asarray(p, float)
    o = np.argsort(p)
    q = np.minimum.accumulate((p[o] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    out = np.empty(len(p))
    out[o] = np.clip(q, 0, 1)
    return out


rk_stats['bh_q'] = rk_stats.groupby('arm').p.transform(bh)
rk_stats.to_csv(LOBO / 'lobo_rkhs_stats.csv', index=False)

HC_C, GOOD_C, WEAK_C, BAD_C, HALF = '#A6A6A6', '#1A9850', "#E98317", '#D73027', 0.42
stars = lambda q: '***' if q < 0.001 else '**' if q < 0.01 else '*' if q < 0.05 else 'n.s.'


def kde_curve(vals, lo, hi):
    k = gaussian_kde(vals, bw_method='scott')
    pad = 2.5 * k.factor * vals.std(ddof=1)
    grid = np.linspace(max(vals.min() - pad, lo), vals.max() + pad, 400)
    return grid, k(grid)


lfolds = json.load(open(config.METHOD_COMP_CACHE_DIR / 'lobo_folds.json'))
n_genes = {'engine': int(np.median([len(pd.read_pickle(LOBO / f['batch_dir'] / 'gene_names.pkl')) for f in lfolds.values()]))}
for impl in ['autoencoder', 'pca']:
    n_genes[impl] = int(np.median([len(frozen_latent.load_fit(impl, f"lobo_{f['batch_dir']}")['genes']) for f in lfolds.values()]))
n_genes['outsingle'] = n_genes['peer'] = n_genes['autoencoder']

order = rk_stats[rk_stats.arm == 'engine'].sort_values('auc').batch.tolist()
fig, axes = plt.subplots(1, len(ARM_LAB), figsize=(2.8 * len(ARM_LAB), 0.8 * len(order) + 1.2), sharey=True)
for ax, (a, lab) in zip(axes, ARM_LAB.items()):
    st_a = rk_stats[rk_stats.arm == a].set_index('batch')
    lo, hi = rk.loc[rk.arm == a, 'rkhs'].quantile([0, 1])
    span = hi - lo
    for y, b in enumerate(order):
        g = rk[(rk.arm == a) & (rk.batch == b)]
        hc, dis = g.loc[g.is_hc, 'rkhs'].values, g.loc[~g.is_hc, 'rkhs'].values
        delta = np.median(dis) - np.median(hc)
        q = st_a.loc[b, 'bh_q']
        dc = BAD_C if delta <= 0 else GOOD_C if q < 0.05 else WEAK_C
        curves = [kde_curve(v, 0.0, None) for v in (hc, dis)]
        top = max(d.max() for _, d in curves)
        for vals, sign, c, (grid, dens) in zip((hc, dis), (1, -1), (HC_C, dc), curves):
            yy = y + sign * HALF * dens / top
            ax.fill_between(grid, y, yy, color=c, alpha=0.55, lw=0)
            ax.plot(grid, yy, color=c, lw=1.0)
            med = np.median(vals)
            ax.plot([med, med], [y, y + sign * HALF * 0.85], color='black', lw=1.1)
        if abs(delta) > 0.03 * span:
            ax.annotate('', xy=(np.median(dis), y), xytext=(np.median(hc), y),
                        arrowprops=dict(arrowstyle='-|>', color='black', lw=1.3, mutation_scale=9, shrinkA=0, shrinkB=0), zorder=4)
        ax.text(0.86, y + 0.05, stars(q), va='bottom', ha='center', transform=ax.get_yaxis_transform(),
                fontsize=FS_ANNOT_LG, fontweight='bold' if q < 0.05 else 'normal', color='black' if q < 0.05 else '0.45')
        ax.text(0.86, y - 0.02, f'Δ = {delta:+.3f}', va='top', ha='center', transform=ax.get_yaxis_transform(),
                fontsize=FS_ANNOT_SM - 1, color='0.2')
    ax.set_xlim(lo - 0.05 * span, hi + 0.75 * span)
    ax.set_title(f'{lab}\n({n_genes[a]:,} genes)', fontsize=FS_ANNOT_LG)
    ax.tick_params(axis='x', labelsize=FS_ANNOT_SM)
    ax.grid(alpha=0.2,  zorder=0)
axes[0].set_yticks(range(len(order)), [b.replace(' et al._Batch_', '_') for b in order])
axes[len(axes) // 2].set_xlabel('RKHS distance to the batch HC mean embedding')
fig.legend([plt.Rectangle((0, 0), 1, 1, fc=HC_C, alpha=0.55), plt.Rectangle((0, 0), 1, 1, fc=GOOD_C, alpha=0.55),
            plt.Rectangle((0, 0), 1, 1, fc=WEAK_C, alpha=0.55), plt.Rectangle((0, 0), 1, 1, fc=BAD_C, alpha=0.55), plt.Line2D([], [], color='black', lw=1.3, marker='>', markevery=[1])],
           ['HC (upper half)', 'Disease farther, BH q < 0.05', 'Disease farther, n.s.', 'Disease closer than HC', r'$\Delta$ = median(disease) $-$ median(HC)'],
           frameon=False, fontsize=FS_ANNOT_SM, ncol=5, loc='lower center', bbox_to_anchor=(0.5, 1.0))
plt.tight_layout()
plt.savefig(FIG / 'lobo_rkhs_disease_vs_hc.png', dpi=300, bbox_inches='tight')
plt.show()
rk_stats.pivot(index='batch', columns='arm', values='bh_q').round(4)

/tmp/ipykernel_1005182/2485317060.py:138: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


arm,autoencoder,engine,outsingle,pca,peer
batch,,,,,
Chen et al._Batch_1,0.0037,0.0145,0.0046,0.0018,0.0057
Moore et al._Batch_1,0.5194,0.0215,0.3527,0.4881,0.2799
Moufarrej et al._Batch_1,0.9836,0.0920,0.7631,0.8467,0.8039
Moufarrej et al._Batch_4,0.8899,0.0215,0.6120,0.8467,0.5756
Ward Z et al._Batch_1,0.0041,0.0098,0.0002,0.0018,0.0001
